# Variational Quantum Classifier

- Install the required libraries
- Import the required libraries
- Generate + pre-process the data
- Define the Quantum Feature Map
- Define the Variational Quantum Circuit
- Define the Quantum Classifier
- Train the VQC model
- Evaluate the model

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification


In [2]:
from qiskit_aer import Aer 
from qiskit.primitives import StatevectorSampler 
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes, zz_feature_map, real_amplitudes #ZZFeatureMap is deprecated
from qiskit_algorithms.optimizers import COBYLA
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import Optimize1qGatesDecomposition, CommutativeCancellation


## Generate + pre-process the data

In [3]:
x,y = make_classification(
    n_samples=100,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42
)

## Normalize the data


In [4]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [5]:
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42)

feature_map = zz_feature_map(feature_dimension=2, reps=2, entanglement='linear')


In [7]:
ansatz = real_amplitudes(num_qubits=2, reps=3, entanglement='linear')

sampler = StatevectorSampler()

In [8]:
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=sampler)

COBYLA : stands for Constrained Optimization BY Linear Approximation.  

In [9]:
optimizer = COBYLA(maxiter=100)
pass_manager = PassManager(
    Optimize1qGatesDecomposition(basis=['u3','cx']),CommutativeCancellation()
)

In [10]:
vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    pass_manager=pass_manager
)

## Train the model

In [12]:
vqc.fit(x_train, y_train)

## Make predictions and evaluate the model

In [13]:
y_test_pred = vqc.predict(x_test)
accuracy = np.mean(y_test_pred == y_test)

In [14]:
print(f"QVC Accuracy: {accuracy:.2f} %")

QVC Accuracy: 0.75 %
